# Обучение моделей cat/dog — оркестратор для ВМ

Запускать **из корня проекта** на ВМ. Каждая ячейка зовёт CLI-шаги из README.

**Что внутри:**
1. установка стека (один раз);
2. напоминание про `clearml-init` (ручной шаг);
3. sanity-check конфига бейзлайна;
4. обучение бейзлайна — DeepLabV3+ R50-d8;
5. анализ качества лучшего чекпойнта на test.

UNet «с нуля» по обоснованию из учебника (см. README Этап 2) сознательно
не запускается. Эксперименты Этапа 3 (аугментации, лосс 1:3, R101-mg124)
добавляются ниже после получения метрики бейзлайна.

## 1. Установка стека (один раз)

In [ ]:
!bash practicum_work/setup_vm.sh

## 2. ClearML credentials

В терминале ВМ выполнить **один раз** интерактивно:
```
clearml-init
```
Вставить creds с https://app.clear.ml/profile.

## 3. Sanity-check (проверка, что всё собирается без обучения)

In [ ]:
!python practicum_work/sanity_check.py configs/deeplabv3plus_practice/deeplabv3plus_r50-d8_1xb16-practice_dataset-256x256.py

## 4. Обучение бейзлайна — DeepLabV3+ R50-d8

~30–45 мин на одном GPU (по бенчмарку из урока 0.077 с/итер × ~1200 итер).
Ссылка на ClearML появится в первых строках вывода.

In [ ]:
!python tools/train.py configs/deeplabv3plus_practice/deeplabv3plus_r50-d8_1xb16-practice_dataset-256x256.py

## 5. Per-image Dice анализ лучшего чекпойнта бейзлайна на test

Скрипт сохраняет CSV `per_image_dice.csv`, плюс по 5 best/worst PNG-триплетов
(image | GT | pred) для отчёта Этапа 2.

In [ ]:
import glob

CFG  = "configs/deeplabv3plus_practice/deeplabv3plus_r50-d8_1xb16-practice_dataset-256x256.py"
NAME = CFG.split("/")[-1].replace(".py", "")
ckpts = sorted(glob.glob(f"work_dirs/{NAME}/best_mDice_epoch_*.pth"))
assert ckpts, f"no best checkpoint for {NAME}"
ckpt = ckpts[-1]
print("==>", NAME, ckpt)

!python practicum_work/src/analysis/per_image_dice.py \
    --config {CFG} \
    --checkpoint {ckpt} \
    --split test \
    --out practicum_work/supplementary/viz/test_baseline_deeplab --n 5